In [14]:
#load and shape (comparison with predictions)
import pandas as pd

df = pd.read_csv("teaching_prices_clean.csv")
df['date'] = pd.to_datetime(df['date'])

#my predictions from the attempt file before comparing it
pred_rows, pred_cols, pred_dates, pred_items = 96, 5, 5, 8

# Actual shape is (rows, columns) and it is compared with my predicted one
print("Shape:", df.shape, "| predicted:", (pred_rows, pred_cols))
# nunique() counts distinct values in a column
print("Distinct dates:", df['date'].nunique(), "| predicted:", pred_dates)
# Which cities exist in the data? My Claim 2 example depends on this:
print("Distinct items:", df['item'].nunique(), "| predicted:", pred_items)
# Column data types; checking that price is numeric and date is a date
print("Cities:", df['city'].unique())
print(df.dtypes)

Shape: (96, 6) | predicted: (96, 5)
Distinct dates: 12 | predicted: 5
Distinct items: 4 | predicted: 8
Cities: <StringArray>
['City 1', 'City 2']
Length: 2, dtype: str
date          datetime64[us]
item                     str
city                     str
price                float64
unit                     str
data_label               str
dtype: object


In [15]:
#skew prediction
# Summary statistics for price: count, mean, min, quartiles, max
print(df['price'].describe())
# Mean vs median; a mean above the median hints at right skew
print("Mean:", df['price'].mean(), "| Median:", df['price'].median())
# Skewness: positive = right-skewed, near 0 = roughly symmetric, negative = left-skewed
print("Skew:", df['price'].skew())
#my prediction about the mean and the skewness were hence both proven right.

count     96.000000
mean     202.166667
std       79.374949
min       99.500000
25%      142.250000
50%      192.250000
75%      234.625000
max      350.000000
Name: price, dtype: float64
Mean: 202.16666666666666 | Median: 192.25
Skew: 0.5689455948692057


In [16]:
#the pivot table
## Pivot: one row per date, one column per (item, city) pair, cells = price
table = df.pivot_table(index='date', columns=['item', 'city'], values='price')
# Counting empty cells since gaps would weaken any calculation across time
print(table)
print("Missing cells:", table.isna().sum().sum())


item        Oil B        Pulse C        Staple A        Sweetener D       
city       City 1 City 2  City 1 City 2   City 1 City 2      City 1 City 2
date                                                                      
2026-01-01  298.0  304.0   200.0  206.0    101.0  107.0       149.0  155.0
2026-01-08  304.0  310.0   202.5  208.5     99.5  105.5       153.5  159.5
2026-01-15  310.0  316.0   200.0  206.0    103.0  109.0       153.0  159.0
2026-01-22  311.0  317.0   202.5  208.5    106.5  112.5       157.5  163.5
2026-01-29  317.0  323.0   200.0  206.0    105.0  111.0       162.0  168.0
2026-02-05  318.0  324.0   202.5  208.5    108.5  114.5       161.5  167.5
2026-02-12  324.0  330.0   205.0  211.0    107.0  113.0       166.0  172.0
2026-02-19  330.0  336.0   202.5  208.5    110.5  116.5       165.5  171.5
2026-02-26  331.0  337.0   205.0  211.0    114.0  120.0       170.0  176.0
2026-03-05  337.0  343.0   202.5  208.5    112.5  118.5       174.5  180.5
2026-03-12  338.0  344.0 

In [17]:
#Questions 1 and 4 from AI, rates of change
# Period-to-period % change for each series (first row is empty because it has no previous date)
changes = table.pct_change() * 100
print(changes.round(2))
# Total % change from first to last date for each series
total_change = (table.iloc[-1] / table.iloc[0] - 1) * 100
#compared the two cities side by side
print(total_change.round(2))

item        Oil B        Pulse C        Staple A        Sweetener D       
city       City 1 City 2  City 1 City 2   City 1 City 2      City 1 City 2
date                                                                      
2026-01-01    NaN    NaN     NaN    NaN      NaN    NaN         NaN    NaN
2026-01-08   2.01   1.97    1.25   1.21    -1.49  -1.40        3.02   2.90
2026-01-15   1.97   1.94   -1.23  -1.20     3.52   3.32       -0.33  -0.31
2026-01-22   0.32   0.32    1.25   1.21     3.40   3.21        2.94   2.83
2026-01-29   1.93   1.89   -1.23  -1.20    -1.41  -1.33        2.86   2.75
2026-02-05   0.32   0.31    1.25   1.21     3.33   3.15       -0.31  -0.30
2026-02-12   1.89   1.85    1.23   1.20    -1.38  -1.31        2.79   2.69
2026-02-19   1.85   1.82   -1.22  -1.18     3.27   3.10       -0.30  -0.29
2026-02-26   0.30   0.30    1.23   1.20     3.17   3.00        2.72   2.62
2026-03-05   1.81   1.78   -1.22  -1.18    -1.32  -1.25        2.65   2.56
2026-03-12   0.30   0.29 

In [18]:
#Question 2, city gaps per item
for item in df['item'].unique():
    # One row per date, one column per city, for this item only
    view = df[df['item'] == item].pivot_table(index='date', columns='city', values='price')
    # Names of the cities (this assumes exactly two)
    cities = list(view.columns)
    # Gap = first city's price minus second city's price on each date
    view['gap'] = view[cities[0]] - view[cities[1]]
    print(item, f"(gap = {cities[0]} minus {cities[1]})")
    print(view.round(2))
    print()

Staple A (gap = City 1 minus City 2)
city        City 1  City 2  gap
date                           
2026-01-01   101.0   107.0 -6.0
2026-01-08    99.5   105.5 -6.0
2026-01-15   103.0   109.0 -6.0
2026-01-22   106.5   112.5 -6.0
2026-01-29   105.0   111.0 -6.0
2026-02-05   108.5   114.5 -6.0
2026-02-12   107.0   113.0 -6.0
2026-02-19   110.5   116.5 -6.0
2026-02-26   114.0   120.0 -6.0
2026-03-05   112.5   118.5 -6.0
2026-03-12   116.0   122.0 -6.0
2026-03-19   114.5   120.5 -6.0

Oil B (gap = City 1 minus City 2)
city        City 1  City 2  gap
date                           
2026-01-01   298.0   304.0 -6.0
2026-01-08   304.0   310.0 -6.0
2026-01-15   310.0   316.0 -6.0
2026-01-22   311.0   317.0 -6.0
2026-01-29   317.0   323.0 -6.0
2026-02-05   318.0   324.0 -6.0
2026-02-12   324.0   330.0 -6.0
2026-02-19   330.0   336.0 -6.0
2026-02-26   331.0   337.0 -6.0
2026-03-05   337.0   343.0 -6.0
2026-03-12   338.0   344.0 -6.0
2026-03-19   344.0   350.0 -6.0

Pulse C (gap = City 1 minus Cit

In [13]:
#average price level by city and date (related to my own Claim 1 and 2)
avg = df.groupby(['date', 'city'])['price'].mean().unstack()
print(avg.round(2))
# Percentage change of that average over time, per city
print((avg.pct_change() * 100).round(2))

city        City 1  City 2
date                      
2026-01-01  187.00  193.00
2026-01-08  189.88  195.88
2026-01-15  191.50  197.50
2026-01-22  194.38  200.38
2026-01-29  196.00  202.00
2026-02-05  197.62  203.62
2026-02-12  200.50  206.50
2026-02-19  202.12  208.12
2026-02-26  205.00  211.00
2026-03-05  206.62  212.62
2026-03-12  208.25  214.25
2026-03-19  211.12  217.12
city        City 1  City 2
date                      
2026-01-01     NaN     NaN
2026-01-08    1.54    1.49
2026-01-15    0.86    0.83
2026-01-22    1.50    1.46
2026-01-29    0.84    0.81
2026-02-05    0.83    0.80
2026-02-12    1.45    1.41
2026-02-19    0.81    0.79
2026-02-26    1.42    1.38
2026-03-05    0.79    0.77
2026-03-12    0.79    0.76
2026-03-19    1.38    1.34
